# EDA -- HTR Error Dataset

Exploratory analysis of the dataset produced by `generate_dataset.py`:
clean/corrupted handwritten word-image pairs, where the corruption is a
real handwritten glyph of one letter spliced over another letter, by the
*same* writer.

This notebook produces descriptive statistics and figures describing the
dataset itself (not model performance -- see `results.ipynb` for that).
All figures are saved as vector PDF and 300dpi PNG under `results/eda/`;
summary tables are exported as `.csv`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from plotting import apply_style, style_ax, save_fig, BLUE, ORANGE, AQUA, INK_MUTED, seq_cmap
from textmetrics import load_english_wordset

apply_style()

DATASET_DIR = Path("dataset")
OUT_DIR = Path("results/eda")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATASET_DIR / "metadata.csv")
df["word_len"] = df["word"].str.len()
print(f"{len(df)} pairs, {df['writer_id'].nunique()} writers")


## 0. Sanity check

Every `(word, error_word)` pair must differ in exactly one character
position -- this is a hard invariant of the generation process, not a
statistical claim, so we assert it rather than just report it.


In [ ]:
def n_diff_chars(a, b):
    return sum(1 for x, y in zip(a, b) if x != y) if len(a) == len(b) else None

df["n_diff_chars"] = [n_diff_chars(w, e) for w, e in zip(df["word"], df["error_word"])]
assert (df["n_diff_chars"] == 1).all(), "found a pair that differs by more than one character!"
print("OK: every pair differs in exactly one character")


## 1. Dataset overview

In [ ]:
per_writer = df["writer_id"].value_counts()
overview = pd.Series({
    "Total pairs": len(df),
    "Unique writers": df["writer_id"].nunique(),
    "Words per writer (mean)": round(per_writer.mean(), 2),
    "Words per writer (std)": round(per_writer.std(), 2),
    "Words per writer (min/max)": f"{per_writer.min()}/{per_writer.max()}",
    "Word length (mean)": round(df["word_len"].mean(), 2),
    "Word length (std)": round(df["word_len"].std(), 2),
    "Word length (range)": f"{df['word_len'].min()}-{df['word_len'].max()}",
    "Distinct source words": df["word"].str.lower().nunique(),
})
overview.to_frame("value")


## 2. Word-length distribution

The pipeline requires clean gaps between letters to segment reliably,
which favours short, more print-like words -- a limitation worth stating
explicitly alongside any use of this dataset.


In [ ]:
counts = df["word_len"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.bar(counts.index, counts.values, color=BLUE, width=0.6)
style_ax(ax)
ax.set_xlabel("Word length (characters)")
ax.set_ylabel("Count")
save_fig(fig, "fig1_word_length", OUT_DIR)
plt.show()


## 3. Position of the altered letter

Joint distribution of word length and the (0-indexed) position of the
swapped letter, to check the random position sampling is not
systematically biased toward word-initial or word-final letters.


In [ ]:
pivot = df.pivot_table(index="word_len", columns="letter_index", values="id", aggfunc="count", fill_value=0)
fig, ax = plt.subplots(figsize=(5, 3.2))
im = ax.imshow(pivot.values, cmap=seq_cmap, aspect="auto")
ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index)
ax.set_xlabel("Letter index altered"); ax.set_ylabel("Word length")
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Count")
save_fig(fig, "fig2_position_heatmap", OUT_DIR)
plt.show()


## 4. Magnitude of the change

Two views of "how much changes": at the **character level** (fraction of
the word's letters altered = 1 / word length) and at the **pixel level**
(relative change in crop width once the replacement glyph is spliced in,
since it is placed at its own natural size rather than stretched to the
old letter's box).


In [ ]:
df["frac_altered"] = 1 / df["word_len"]

sizes = df["clean_image"].apply(lambda p: Image.open(DATASET_DIR / p).size)
df["clean_w"], df["clean_h"] = zip(*sizes)
err_sizes = df["error_image"].apply(lambda p: Image.open(DATASET_DIR / p).size)
df["error_w"], df["error_h"] = zip(*err_sizes)
df["width_change_pct"] = (df["error_w"] - df["clean_w"]) / df["clean_w"] * 100

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
frac_counts = (df["frac_altered"] * 100).round(1).value_counts().sort_index()
axes[0].bar([f"{v:.0f}%" for v in frac_counts.index], frac_counts.values, color=BLUE, width=0.6)
style_ax(axes[0])
axes[0].set_xlabel("Word altered (1 / word length)"); axes[0].set_ylabel("Count")

axes[1].hist(df["width_change_pct"], bins=30, color=ORANGE)
style_ax(axes[1])
axes[1].axvline(0, color=INK_MUTED, linewidth=1, linestyle="--")
axes[1].set_xlabel("Image width change after splicing (%)"); axes[1].set_ylabel("Count")

save_fig(fig, "fig3_magnitude", OUT_DIR)
plt.show()
print(df["width_change_pct"].describe())


## 5. Letter-substitution confusion matrix

Rows: the letter that was removed. Columns: the letter spliced in. Because
replacement glyphs are drawn from the same writer's own alphabet, this
matrix reflects each writer's available letters more than any deliberate
visual-confusability design -- reported here for transparency.


In [ ]:
conf = pd.crosstab(df["original_letter"].str.lower(), df["new_letter"].str.lower())
letters_order = sorted(set(conf.index) | set(conf.columns))
conf = conf.reindex(index=letters_order, columns=letters_order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(conf.values, cmap=seq_cmap)
ax.set_xticks(range(len(letters_order))); ax.set_xticklabels(letters_order, fontsize=8)
ax.set_yticks(range(len(letters_order))); ax.set_yticklabels(letters_order, fontsize=8)
ax.set_xlabel("Substituted-in letter"); ax.set_ylabel("Original letter")
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Count")
save_fig(fig, "fig4_confusion", OUT_DIR)
plt.show()


In [ ]:
out_counts = df["original_letter"].str.lower().value_counts().head(10)
in_counts = df["new_letter"].str.lower().value_counts().head(10)
letters5 = sorted(set(out_counts.index) | set(in_counts.index))
out_vals = [out_counts.get(l, 0) for l in letters5]
in_vals = [in_counts.get(l, 0) for l in letters5]
x = np.arange(len(letters5))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(x - 0.2, out_vals, width=0.4, color=BLUE, label="Removed (original)")
ax.bar(x + 0.2, in_vals, width=0.4, color=ORANGE, label="Inserted (new)")
ax.set_xticks(x); ax.set_xticklabels(letters5)
style_ax(ax)
ax.legend(frameon=False)
ax.set_ylabel("Count")
save_fig(fig, "fig5_top_letters", OUT_DIR)
plt.show()


## 6. Letter frequency vs. reference English corpus

Sanity check that the *source* words (before corruption) follow a roughly
natural English letter distribution -- reference frequencies from Lewand
(2000), a commonly cited corpus-frequency table.


In [ ]:
REF_FREQ = {'e': 12.7, 't': 9.1, 'a': 8.2, 'o': 7.5, 'i': 7.0, 'n': 6.7, 's': 6.3, 'h': 6.1,
    'r': 6.0, 'd': 4.3, 'l': 4.0, 'c': 2.8, 'u': 2.8, 'm': 2.4, 'w': 2.4, 'f': 2.2,
    'g': 2.0, 'y': 2.0, 'p': 1.9, 'b': 1.5, 'v': 1.0, 'k': 0.8, 'j': 0.15, 'x': 0.15,
    'q': 0.10, 'z': 0.07}

all_letters = pd.Series(list("".join(df["word"].str.lower())))
dataset_freq = all_letters.value_counts(normalize=True) * 100
letters6 = sorted(REF_FREQ.keys(), key=lambda l: -REF_FREQ[l])
ds_vals = [dataset_freq.get(l, 0) for l in letters6]
ref_vals = [REF_FREQ[l] for l in letters6]
x = np.arange(len(letters6))

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(x - 0.2, ds_vals, width=0.4, color=BLUE, label="This dataset (source words)")
ax.bar(x + 0.2, ref_vals, width=0.4, color=ORANGE, label="Reference English corpus (Lewand, 2000)")
ax.set_xticks(x); ax.set_xticklabels(letters6)
style_ax(ax)
ax.legend(frameon=False)
ax.set_ylabel("Frequency (%)")
save_fig(fig, "fig6_letter_frequency", OUT_DIR)
plt.show()


## 7. Real words vs. non-words (linguistic-prior relevance)

How often does the single-letter swap happen to land on *another real
English word*? Those cases are the hardest test of visual grounding, since
a language-prior-driven system has a plausible real word to fall back on
instead of the one actually written.

Caveat: `nltk.corpus.words` is large (~235k entries) and includes
archaic/rare forms, so this over-counts "real word" slightly -- worth a
manual spot check of borderline cases before citing the exact percentage.


In [ ]:
wordset = load_english_wordset()
df["word_is_dict"] = df["word"].str.lower().isin(wordset)
df["error_is_dict"] = df["error_word"].str.lower().isin(wordset)

fig, ax = plt.subplots(figsize=(4, 3.2))
vals = [df["word_is_dict"].mean() * 100, df["error_is_dict"].mean() * 100]
ax.bar(["Original word", "Corrupted word"], vals, color=[BLUE, ORANGE], width=0.5)
style_ax(ax)
ax.set_ylabel("In English dictionary (%)")
ax.set_ylim(0, 100)
save_fig(fig, "fig7_realword", OUT_DIR)
plt.show()

print(f"Original word in dictionary: {df['word_is_dict'].mean():.1%}")
print(f"Corrupted word in dictionary: {df['error_is_dict'].mean():.1%}")
print(f"Both real (minimal-pair) cases: {(df['word_is_dict'] & df['error_is_dict']).mean():.1%}")


In [ ]:
minimal_pairs = df[df["word_is_dict"] & df["error_is_dict"]][["word", "error_word", "writer_id"]]
minimal_pairs.to_csv(OUT_DIR / "minimal_pair_examples.csv", index=False)
minimal_pairs.head(15)


## 8. Image geometry

Pixel dimensions of the (clean) word crops -- relevant for anyone choosing
a resize/patch strategy before feeding the dataset to a model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(df["clean_w"], bins=25, color=BLUE)
style_ax(axes[0]); axes[0].set_xlabel("Clean image width (px)"); axes[0].set_ylabel("Count")
axes[1].hist(df["clean_h"], bins=25, color=AQUA)
style_ax(axes[1]); axes[1].set_xlabel("Clean image height (px)"); axes[1].set_ylabel("Count")
save_fig(fig, "fig8_image_geometry", OUT_DIR)
plt.show()


## 9. Writer distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.hist(per_writer.values, bins=range(1, per_writer.max() + 2), color=BLUE, align="left")
style_ax(ax)
ax.set_xlabel("Samples per writer"); ax.set_ylabel("Number of writers")
save_fig(fig, "fig9_writer_distribution", OUT_DIR)
plt.show()
per_writer.describe()


## 10. Qualitative examples

A random sample of clean/corrupted pairs (top: clean, bottom: corrupted)
-- useful for visually judging splice quality.


In [ ]:
rng = np.random.default_rng(7)
sample_idx = rng.choice(len(df), size=9, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(9, 6))
for ax, i in zip(axes.flat, sample_idx):
    row = df.iloc[i]
    c = Image.open(DATASET_DIR / row["clean_image"]).convert("L")
    e = Image.open(DATASET_DIR / row["error_image"]).convert("L")
    h = max(c.height, e.height)
    w = max(c.width, e.width)
    combo = Image.new("L", (w, h * 2 + 6), 255)
    combo.paste(c, (0, 0))
    combo.paste(e, (0, h + 6))
    ax.imshow(np.array(combo), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{row['word']} \u2192 {row['error_word']}", fontsize=10)
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "fig10_sample_grid", OUT_DIR)
plt.show()


## 11. Summary table

Exported to `results/eda/summary_table.csv`.


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total word pairs", "Unique writers", "Words per writer (mean +/- std)",
        "Word length, characters (mean +/- std)", "Original word in dictionary",
        "Corrupted word in dictionary", "Minimal-pair (both real words)",
        "Mean image width change after splicing", "Distinct source words",
    ],
    "Value": [
        f"{len(df)}",
        f"{df['writer_id'].nunique()}",
        f"{per_writer.mean():.1f} +/- {per_writer.std():.1f}",
        f"{df['word_len'].mean():.1f} +/- {df['word_len'].std():.1f}",
        f"{df['word_is_dict'].mean():.1%}",
        f"{df['error_is_dict'].mean():.1%}",
        f"{(df['word_is_dict'] & df['error_is_dict']).mean():.1%}",
        f"{df['width_change_pct'].mean():+.1f}%",
        f"{df['word'].str.lower().nunique()}",
    ],
})
summary.to_csv(OUT_DIR / "summary_table.csv", index=False)
summary


## Notes

- **Short-word skew**: most pairs come from 3-4 letter words (section 2),
  a direct consequence of requiring clean whitespace gaps for letter
  segmentation. Longer cursive words are under-represented.
- **Substitution pool is per-writer, not adversarially chosen**: the
  replacement letter is drawn from whatever that writer happened to
  produce elsewhere (section 5), not selected for visual similarity to the
  original letter.
- **A meaningful fraction of corruptions land on another real word**
  (section 7) -- useful for splitting an evaluation set into "non-word" vs
  "real-word" conditions.
- **Pixel-level change is asymmetric**: corrupted images are, on average,
  slightly *wider* than clean ones (section 4), since replacement glyphs
  are placed at their own natural size rather than stretched into the old
  letter's box -- worth noting for any width-normalised metric.
- All figures/tables in `results/eda/` are regenerated by re-running this
  notebook against `dataset/metadata.csv`; no manual post-editing applied.
